# CEP — cleaning / adjust step (raw → analysis-ready)

**One transformation only.** Loads the untouched EDQM CEP dump (`raw/EXPORT_WEB_CEP.txt`),
and produces two analysis-ready tables for the CEP↔trade match (parallel to BfArM):

- `cep_antibiotics_clean.csv` — one row per CEP in the 4 antibiotic HS-6 classes (long).
- `cep_origins_by_class.csv` — per HS class → certified origin countries + CEP counts.

Every step is explicit and eyeball-able (country map, substance→class keyword map). Nothing is
dropped silently: unmapped countries / substances are printed for review.

> Provenance & column notes: see `README.md` in this folder.

## 0 · Setup

In [1]:
import pandas as pd, numpy as np, re
from pathlib import Path

HERE = Path.cwd()                      # run from Data/CEP
RAW  = HERE / 'raw' / 'EXPORT_WEB_CEP.txt'
assert RAW.exists(), f'raw dump not found: {RAW}'

HS    = ['294110', '294130', '294150', '294190']
HSLAB = {'294110': 'Penicillins', '294130': 'Tetracyclines',
         '294150': 'Macrolides', '294190': 'Other antibiotics'}
HSATC = {'294110': 'J01C', '294130': 'J01AA', '294150': 'J01FA',
         '294190': 'J01D / J01G / J01FF / J01XA / J01XB'}
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 80)

## 1 · Load the raw dump (verbatim)

In [2]:
raw = pd.read_csv(RAW, sep='\t', dtype=str, encoding='utf-8-sig')
print('shape:', raw.shape)
print('columns:', list(raw.columns))
print()
print('Type CEP:');   print(raw['Type CEP'].value_counts())
print()
print('Status CEP:'); print(raw['Status CEP'].value_counts())
raw.head(3)

shape: (9388, 11)
columns: ['Monograph Number', 'Substance', 'Type CEP', 'Certificate (CEP) Holder', 'Holder SPOR ORG-ID / SPOR LOC-ID', 'Certificate (CEP) Number', 'Issue Date CEP', 'Status CEP', 'Renewal due', 'End date CEP', 'Closure Date of last Procedure']

Type CEP:
Type CEP
Chemical            8512
TSE                  689
Chemical and TSE     115
Herbal                72
Name: count, dtype: int64

Status CEP:
Status CEP
Valid                                         7493
Withdrawn by Holder                           1299
Expired                                        361
Withdrawn by EDQM Failure to CEP procedure     180
Suspended by EDQM GMP non-compliance            32
Suspended by Holder                             14
Withdrawn by EDQM GMP non-compliance             9
Name: count, dtype: int64


,Monograph Number,Substance,Type CEP,Certificate (CEP) Holder,Holder SPOR ORG-ID / SPOR LOC-ID,Certificate (CEP) Number,Issue Date CEP,Status CEP,Renewal due,End date CEP,Closure Date of last Procedure
0,0,"(6,7)-3-hydroxymethyl-7-(z-2-methoxyimino-2-(fur-2-yl)acetamido)ceph-3-em-4-...",TSE,Glaxo Wellcome London GB,NaN,R0-CEP 2000-275 - Rev 01,21/11/2001,Withdrawn by Holder,NaN,03/08/2006,03/08/2006
1,0,"1,2-dihydrotriamcinolone",TSE,Pharmacia & Upjohn Company Kalamazoo US,NaN,R0-CEP 2001-231 - Rev 00,09/04/2002,Expired,NaN,09/04/2007,15/06/2007
2,0,16-alpha-Hydroxyprednisolone,TSE,Pharmacia & Upjohn Company Kalamazoo US,NaN,R0-CEP 2001-018 - Rev 00,24/10/2001,Withdrawn by Holder,NaN,24/10/2006,24/10/2006


## 2 · Parse the holder country → ISO-3

The holder country is the **last token** of `Certificate (CEP) Holder` (ISO alpha-2), e.g.
`Glaxo Wellcome London GB` → `GB`. We extract it and convert to ISO-3 (BACI codes). Any holder
string without a trailing code is handled by an explicit override; anything still unmapped is
printed so nothing is dropped silently.

In [3]:
# ISO alpha-2 -> alpha-3 (only the codes that occur; extend if the dump grows)
ISO2_3 = {
 'AT':'AUT','BE':'BEL','BR':'BRA','CH':'CHE','CN':'CHN','DE':'DEU','DK':'DNK','EG':'EGY',
 'ES':'ESP','FR':'FRA','GB':'GBR','HR':'HRV','HU':'HUN','IE':'IRL','IL':'ISR','IN':'IND',
 'IT':'ITA','JP':'JPN','KR':'KOR','MO':'MAC','MX':'MEX','MY':'MYS','NL':'NLD','OM':'OMN',
 'PL':'POL','PR':'PRI','PT':'PRT','SE':'SWE','SG':'SGP','SI':'SVN','SK':'SVK','TR':'TUR',
 'TW':'TWN','US':'USA',
}
# holder strings that carry no trailing 2-letter code -> fix by hand (city tells the country)
HOLDER_OVERRIDE = {'CJ Corp. Seoul': 'KR'}

def holder_iso2(h):
    h = str(h).strip()
    if h in HOLDER_OVERRIDE: return HOLDER_OVERRIDE[h]
    m = re.search(r'\b([A-Z]{2})\s*$', h)
    return m.group(1) if m else None

raw['iso2'] = raw['Certificate (CEP) Holder'].map(holder_iso2)
raw['iso3'] = raw['iso2'].map(ISO2_3)

bad_iso2 = raw.loc[raw.iso2.isna(), 'Certificate (CEP) Holder'].dropna().unique()
bad_iso3 = sorted(raw.loc[raw.iso2.notna() & raw.iso3.isna(), 'iso2'].unique())
print('holders with no parseable country:', list(bad_iso2)[:20], '...' if len(bad_iso2)>20 else '')
print('alpha-2 codes present but not in ISO2_3 map (extend it!):', bad_iso3)

holders with no parseable country: ['Pfizer Ireland Pharmaceuticals Dun Laoghaire', 'High Gel Co Ltd Yongin Kyunggi-Do', 'DIFC CO. LTD. Ansan-City', 'Choongwae Pharma Corporation Seoul', 'Servier (Ireland) Industries Ltd Arklow'] 
alpha-2 codes present but not in ISO2_3 map (extend it!): ['AE', 'AR', 'AU', 'BG', 'CA', 'CO', 'CZ', 'FI', 'HK', 'ID', 'IS', 'JO', 'LU', 'LV', 'MA', 'MC', 'MH', 'MT', 'NO', 'NZ', 'PK', 'RO', 'RU', 'SA', 'TH', 'UA', 'ZA']


## 3 · Classify each substance into an HS-6 antibiotic class

CEP substance names are chemical/brand-ish (`amoxicillin trihydrate, india, process o`), so we
classify by an **explicit keyword/regex map per class** — verify it visually, exactly like the
`AUTH2ISO` tables in the register notebook. A substance may match **at most one** class; conflicts
are flagged.

In [4]:
PAT = {
 '294110': [r'cillin\b', r'penicillin', r'mecillinam'],                         # penicillins  J01C
 '294130': [r'cycline\b'],                                                      # tetracyclines J01AA
 '294150': [r'thromycin\b', r'spiramycin', r'josamycin', r'midecamycin',        # macrolides   J01FA
            r'oleandomycin', r'rokitamycin', r'miocamycin', r'flurithromycin'],
 '294190': [r'\bcef', r'\bceph', r'clindamycin', r'lincomycin',                 # other J01D/G/FF/XA/XB
            r'vancomycin', r'teicoplanin', r'\w+vancin\b',                      #   glycopeptides
            r'colistin', r'polymyxin',                                          #   polymyxins
            r'gentamicin', r'netilmicin', r'sisomicin', r'tobramycin',          #   aminoglycosides
            r'\w*micin\b', r'\w*kacin\b', r'neomycin', r'streptomycin',
            r'kanamycin', r'paromomycin', r'framycetin', r'spectinomycin',
            r'ribostamycin', r'plazomicin'],
}

def classify(name):
    s = str(name).lower()
    hits = [hs for hs, pats in PAT.items() if any(re.search(p, s) for p in pats)]
    return hits

raw['hs_hits'] = raw['Substance'].map(classify)
raw['n_hits']  = raw['hs_hits'].map(len)

conflicts = raw.loc[raw.n_hits > 1, 'Substance'].unique()
print('CEP rows matching an antibiotic class :', int((raw.n_hits >= 1).sum()))
print('rows matching >1 class (should be 0)  :', int((raw.n_hits > 1).sum()))
if len(conflicts): print('CONFLICTS to resolve:', list(conflicts))

raw['hs'] = raw['hs_hits'].map(lambda h: h[0] if len(h) == 1 else None)

CEP rows matching an antibiotic class : 622
rows matching >1 class (should be 0)  : 0


### 3.1 · Eyeball the classification — distinct substances per class

In [5]:
for hs in HS:
    subs = sorted(raw.loc[raw.hs == hs, 'Substance'].str.lower()
                     .str.split(',').str[0].str.strip().unique())
    print(f'=== {hs}  {HSLAB[hs]}  ({HSATC[hs]}) — {len(subs)} distinct base substances ===')
    print('   ' + ', '.join(subs))
    print()

=== 294110  Penicillins  (J01C) — 23 distinct base substances ===
   amoxicillin sodium, amoxicillin trihydrate, ampicillin, ampicillin sodium, ampicillin trihydrate, bacampicillin hydrochloride, benzylpenicillin (benzathine) tetrahydrate, benzylpenicillin (procaine) monohydrate, benzylpenicillin potassium, benzylpenicillin sodium, cloxacillin sodium, dicloxacillin sodium monohydrate, flucloxacillin magnesium octahydrate, flucloxacillin sodium monohydrate, oxacillin sodium monohydrate, phenoxymethylpenicillin (benzathine) tetrahydrate, phenoxymethylpenicillin potassium, piperacillin monohydrate, piperacillin sodium, pivampicillin, pivmecillinam hydrochloride, seed agar, sultamicillin tosilate dihydrate

=== 294130  Tetracyclines  (J01AA) — 10 distinct base substances ===
   chlortetracycline hydrochloride, demeclocycline hydrochloride, doxycycline hyclate, doxycycline monohydrate, lymecycline, minocycline hydrochloride dihydrate, oxytetracycline dihydrate, oxytetracycline hydrochloride

## 4 · Build the analysis-ready tables

Filter to the 4 classes, keep the fields the match needs, and add a `is_valid` flag (Status = Valid)
so the match can choose valid-only or all-status. We keep **every CEP row** (a substance can have many
CEPs from different holders/countries — that granularity is the point).

In [6]:
keep = raw[raw.hs.notna()].copy()
keep['is_valid']    = keep['Status CEP'].eq('Valid')
keep['substance_base'] = keep['Substance'].str.lower().str.split(',').str[0].str.strip()

clean = keep.rename(columns={
    'Substance':'substance', 'Type CEP':'type_cep', 'Certificate (CEP) Holder':'holder',
    'Certificate (CEP) Number':'cep_number', 'Issue Date CEP':'issue_date',
    'Status CEP':'status', 'End date CEP':'end_date', 'Monograph Number':'monograph',
})[['hs','substance_base','substance','holder','iso2','iso3','type_cep','cep_number',
    'status','is_valid','issue_date','end_date','monograph']].copy()
clean['hs_label'] = clean['hs'].map(HSLAB)

# report anything that would be lost in the match (country not resolved)
lost = clean[clean.iso3.isna()]
print('rows in the 4 classes:', len(clean), '| with unresolved iso3 (excluded from origin sets):', len(lost))
if len(lost): display(lost[['hs','substance','holder','iso2']].head(20))

OUT = HERE / 'cep_antibiotics_clean.csv'
clean.to_csv(OUT, index=False)
print('wrote', OUT, clean.shape)
clean.head(8)

rows in the 4 classes: 622 | with unresolved iso3 (excluded from origin sets): 0
wrote /Users/alessandropesce/Desktop/Drugrs_supply_chain/Data/CEP/cep_antibiotics_clean.csv (622, 14)


,hs,substance_base,substance,holder,iso2,iso3,type_cep,cep_number,status,is_valid,issue_date,end_date,monograph,hs_label
0,294190,(6,"(6,7)-3-hydroxymethyl-7-(z-2-methoxyimino-2-(fur-2-yl)acetamido)ceph-3-em-4-...",Glaxo Wellcome London GB,GB,GBR,TSE,R0-CEP 2000-275 - Rev 01,Withdrawn by Holder,False,21/11/2001,03/08/2006,0,Other antibiotics
148,294190,ceftizoxime sodium,Ceftizoxime sodium,Fujisawa Pharmaceutical Co Ltd Osaka JP,JP,JPN,TSE,R0-CEP 2001-072 - Rev 02,Expired,False,28/02/2005,03/05/2007,0,Other antibiotics
352,294190,neomycin assay agar,"Neomycin Assay Agar, (Antibiotic Medium 11, Product codes 210977 and 210980)","Becton, Dickinson and Company Sparks US",US,USA,TSE,R0-CEP 2000-357 - Rev 02,Withdrawn by Holder,False,22/02/2005,11/07/2006,0,Other antibiotics
434,294110,seed agar,"Seed Agar, Penicillin Assay, (Product codes: 210937, 210938 and 210939)","Becton, Dickinson and Company Sparks US",US,USA,TSE,R0-CEP 2000-380 - Rev 01,Expired,False,22/02/2005,02/12/2007,0,Penicillins
595,294190,streptomycin sulfate,Streptomycin sulfate,"North China Pharmaceutical Huasheng Co., Ltd. Shijiazhuang CN",CN,CHN,Chemical,CEP 2001-394 - Rev 04,Valid,True,16/02/2026,NaN,53,Other antibiotics
596,294190,streptomycin sulfate,Streptomycin sulfate,HEBEI SHENGXUE DACHENG PHARMACEUTICAL CO. LTD. Shijiazhuang CN,CN,CHN,Chemical,CEP 2004-052 - Rev 04,Valid,True,10/04/2026,NaN,53,Other antibiotics
726,294110,benzylpenicillin potassium,Benzylpenicillin potassium,FERSINSA GB S.A. DE C.V. Ramos Arizpe MX,MX,MEX,Chemical,R1-CEP 2001-272 - Rev 05,Valid,True,13/11/2018,NaN,113,Penicillins
727,294110,benzylpenicillin potassium,Benzylpenicillin potassium,"INNER MONGOLIA CHANGSHENG PHARMACEUTICAL CO., LTD. Tuoketuo CN",CN,CHN,Chemical,R0-CEP 2020-163 - Rev 00,Valid,True,10/08/2021,NaN,113,Penicillins


### 4.1 · Certified origins per HS class (what the match consumes)

In [7]:
val = clean[clean.iso3.notna() & clean.is_valid]     # valid CEPs only for the origin set
origins = (val.groupby(['hs','hs_label','iso3'])
              .agg(n_cep=('cep_number','nunique'),
                   n_substances=('substance_base','nunique'),
                   n_holders=('holder','nunique')).reset_index()
              .sort_values(['hs','n_cep'], ascending=[True, False]))
OUT2 = HERE / 'cep_origins_by_class.csv'
origins.to_csv(OUT2, index=False)
print('wrote', OUT2, origins.shape)
origins.head(25)

wrote /Users/alessandropesce/Desktop/Drugrs_supply_chain/Data/CEP/cep_origins_by_class.csv (41, 6)


,hs,hs_label,iso3,n_cep,n_substances,n_holders
1,294110,Penicillins,CHN,30,8,15
4,294110,Penicillins,IND,18,12,1
5,294110,Penicillins,ITA,14,9,4
0,294110,Penicillins,AUT,13,10,1
8,294110,Penicillins,NLD,7,5,1
2,294110,Penicillins,ESP,5,5,2
7,294110,Penicillins,MEX,4,4,1
3,294110,Penicillins,GBR,2,1,1
6,294110,Penicillins,KOR,1,1,1
9,294110,Penicillins,SVK,1,1,1


## 5 · Sanity — the masking signal

Share of **valid** CEPs by origin, EU-27 vs extra-EU, per class. If the de-masking story holds, the
certified API origin for these antibiotics is dominated by China/India — visible here before any
trade matching.

In [8]:
EU27 = {'AUT','BEL','BGR','HRV','CYP','CZE','DNK','EST','FIN','FRA','DEU','GRC','HUN','IRL',
        'ITA','LVA','LTU','LUX','MLT','NLD','POL','PRT','ROU','SVK','SVN','ESP','SWE'}
tab = (val.assign(bloc=np.where(val.iso3.isin(EU27), 'EU-27', 'extra-EU'))
          .groupby(['hs_label','bloc']).cep_number.nunique().unstack(fill_value=0))
tab['extra-EU %'] = (100*tab.get('extra-EU',0)/tab.sum(axis=1)).round(0)
print('valid CEPs by origin bloc, per class:'); display(tab)

top = (val.groupby(['hs_label','iso3']).cep_number.nunique()
          .sort_values(ascending=False).groupby(level=0).head(4))
print('\ntop certified origins per class (valid CEPs):'); print(top)

valid CEPs by origin bloc, per class:


bloc,EU-27,extra-EU,extra-EU %
hs_label,,,
Macrolides,9,44,83.0
Other antibiotics,45,145,76.0
Penicillins,40,55,58.0
Tetracyclines,13,38,75.0



top certified origins per class (valid CEPs):
hs_label           iso3
Other antibiotics  CHN     97
                   IND     33
Tetracyclines      CHN     31
Penicillins        CHN     30
Other antibiotics  ITA     23
Macrolides         CHN     22
                   IND     18
Penicillins        IND     18
                   ITA     14
                   AUT     13
Other antibiotics  KOR     11
Tetracyclines      PRT      6
Macrolides         ESP      5
Tetracyclines      IND      4
Macrolides         NLD      3
Tetracyclines      ITA      3
Name: cep_number, dtype: int64
